In [28]:
import os
from google.cloud import aiplatform
from dotenv import load_dotenv

In [4]:
load_dotenv() 

True

In [5]:
PROJECT_ID = os.environ["PROJECT_ID"]
LOCATION = os.environ["LOCATION"]

In [6]:
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = r"./service_account.json"

In [7]:
aiplatform.init(
    project=PROJECT_ID,
    location=LOCATION,
)

### Container Image

In [2]:
CONTAINER_URI="us-docker.pkg.dev/deeplearning-platform-release/gcr.io/huggingface-pytorch-inference-cu121.2-2.transformers.4-44.ubuntu2204.py311"

### Upload Model

In [10]:
model = aiplatform.Model.upload(
    display_name="distilbert--distilbert-base-uncased-finetuned-sst-2-english",
    serving_container_image_uri=CONTAINER_URI,
    serving_container_environment_variables={
        "HF_MODEL_ID": "distilbert/distilbert-base-uncased-finetuned-sst-2-english",
        "HF_TASK": "text-classification",
    },
)
model.wait()

Creating Model
Create Model backing LRO: projects/344969539300/locations/us-central1/models/5577654262428598272/operations/270487290726318080
Model created. Resource name: projects/344969539300/locations/us-central1/models/5577654262428598272@1
To use this Model in another session:
model = aiplatform.Model('projects/344969539300/locations/us-central1/models/5577654262428598272@1')


### Check Model

In [11]:
models = aiplatform.Model.list()
if not models:
    print("No models found in the Model Registry.")
else:
    print(f"Found {len(models)} models:")
    for item in models:
        print(f"Name: {item.display_name}")
        print(f"ID: {item.resource_name}")
        print(f"Version: {item.version_id}")
        print("-" * 30)

Found 1 models:
Name: distilbert--distilbert-base-uncased-finetuned-sst-2-english
ID: projects/344969539300/locations/us-central1/models/5577654262428598272
Version: 1
------------------------------


### Endpoint

In [12]:
endpoint = aiplatform.Endpoint.create(display_name="distilbert--distilbert-base-uncased-finetuned-sst-2-english-endpoint")

Creating Endpoint
Create Endpoint backing LRO: projects/344969539300/locations/us-central1/endpoints/8224134770020319232/operations/5836936430156251136
Endpoint created. Resource name: projects/344969539300/locations/us-central1/endpoints/8224134770020319232
To use this Endpoint in another session:
endpoint = aiplatform.Endpoint('projects/344969539300/locations/us-central1/endpoints/8224134770020319232')


In [13]:
# Check endpoint
endpoints = aiplatform.Endpoint.list()

for item in endpoints:
    print("Display Name:", item.display_name)
    print("Resource Name:", item.resource_name)
    print("Endpoint ID:", item.name)
    print("-" * 50)

Display Name: distilbert--distilbert-base-uncased-finetuned-sst-2-english-endpoint
Resource Name: projects/344969539300/locations/us-central1/endpoints/8224134770020319232
Endpoint ID: 8224134770020319232
--------------------------------------------------


In [15]:
# Check endpoint
endpoint = aiplatform.Endpoint(
    endpoint_name="projects/344969539300/locations/us-central1/endpoints/8224134770020319232"
)
print("Display Name:", endpoint.display_name)
print("Deployed Models:", endpoint.gca_resource.deployed_models)

Display Name: distilbert--distilbert-base-uncased-finetuned-sst-2-english-endpoint
Deployed Models: []


### Deploy Endpoint

In [29]:
deployed_model = model.deploy(
    endpoint=endpoint,
    machine_type="g2-standard-12",
    accelerator_type="NVIDIA_L4",
    accelerator_count=1,
)

Deploying model to Endpoint : projects/344969539300/locations/us-central1/endpoints/8224134770020319232
Deploy Endpoint model backing LRO: projects/344969539300/locations/us-central1/endpoints/8224134770020319232/operations/4081658475388600320
Endpoint model deployed. Resource name: projects/344969539300/locations/us-central1/endpoints/8224134770020319232


### Simple Inference

In [30]:
output = deployed_model.predict(instances=["I love this product", "I hate this product"], parameters={"top_k": 2})

In [31]:
output

Prediction(predictions=[[{'score': 0.9998788833618164, 'label': 'POSITIVE'}, {'score': 0.0001210561968036927, 'label': 'NEGATIVE'}], [{'score': 0.9997544884681702, 'label': 'NEGATIVE'}, {'score': 0.0002454846107866615, 'label': 'POSITIVE'}]], deployed_model_id='50155872168443904', metadata=None, model_version_id='1', model_resource_name='projects/344969539300/locations/us-central1/models/5577654262428598272', explanations=None)

In [32]:
print(output.predictions[0])

[{'score': 0.9998788833618164, 'label': 'POSITIVE'}, {'score': 0.0001210561968036927, 'label': 'NEGATIVE'}]


### Clean

In [35]:
deployed_model.undeploy_all()
deployed_model.delete()
model.delete()

Undeploying Endpoint model: projects/344969539300/locations/us-central1/endpoints/8224134770020319232
Undeploy Endpoint model backing LRO: projects/344969539300/locations/us-central1/endpoints/8224134770020319232/operations/1536869599226626048
Endpoint model undeployed. Resource name: projects/344969539300/locations/us-central1/endpoints/8224134770020319232
Deleting Endpoint : projects/344969539300/locations/us-central1/endpoints/8224134770020319232
Endpoint deleted. . Resource name: projects/344969539300/locations/us-central1/endpoints/8224134770020319232
Deleting Endpoint resource: projects/344969539300/locations/us-central1/endpoints/8224134770020319232
Delete Endpoint backing LRO: projects/344969539300/locations/us-central1/operations/4156812294170345472
Endpoint resource projects/344969539300/locations/us-central1/endpoints/8224134770020319232 deleted.
Deleting Model : projects/344969539300/locations/us-central1/models/5577654262428598272
Model deleted. . Resource name: projects/3

In [39]:
# endpoint.undeploy_all()

In [38]:
# endpoint.delete(force=True)